In [30]:
## Ada Boost Classifier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings(action='ignore')

%matplotlib inline

In [31]:
df = pd.read_csv('Travel.csv')
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [32]:
df['Gender'].value_counts()

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [33]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [34]:
df['TypeofContact'].value_counts()

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64

In [35]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')

In [36]:
df['Gender'].value_counts()

Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [37]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [38]:
## Missing value based on Percentage
features_with_na = [features for features in df.columns if df[features].isnull().sum() >= 1]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean()*100, 2), 'Missing value')

Age 4.62 Missing value
TypeofContact 0.51 Missing value
DurationOfPitch 5.14 Missing value
NumberOfFollowups 0.92 Missing value
PreferredPropertyStar 0.53 Missing value
NumberOfTrips 2.86 Missing value
NumberOfChildrenVisiting 1.35 Missing value
MonthlyIncome 4.77 Missing value


In [39]:
df[features_with_na].select_dtypes(exclude= 'object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


In [40]:
df.PreferredPropertyStar.mode()

0    3.0
Name: PreferredPropertyStar, dtype: float64

## Imputing Null values
1. Impute Median value for Age column
2. Impute Mode for Type of Contract
3. Impute Median for Duration of Pitch
4. Impute Mode for NumberofFollowup as it is Discrete feature
5. Impute Mode for PreferredPropertyStar
6. Impute Median for NumberofTrips
7. Impute Mode for NumberOfChildrenVisiting
8. Impute Median for MonthlyIncome

In [41]:
df.Age.fillna(df.Age.median(), inplace=True)

df.TypeofContact.fillna(df.TypeofContact.mode()[0], inplace=True)

df.DurationOfPitch.fillna(df.DurationOfPitch.median(), inplace=True)
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0], inplace=True)
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0], inplace=True)
df.NumberOfTrips.fillna(df.NumberOfTrips.median(), inplace=True)
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0], inplace=True)
df.MonthlyIncome.fillna(df.MonthlyIncome.median(), inplace=True)

In [42]:
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [43]:
df.drop('CustomerID', inplace=True, axis=1)

## Feature Engineering


In [44]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [45]:
## Create new column

df['TotalVisiting'] = df['NumberOfPersonVisiting'] + df['NumberOfChildrenVisiting']
df.drop(columns=['NumberOfPersonVisiting', 'NumberOfChildrenVisiting'], axis=1, inplace=True)

In [47]:
## Get all numeric, categorical, discrete, continous feature

numerical_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('No. of Numeric Feature', len(numerical_features))

categorical_feature = [feature for feature in df.columns if df[feature].dtype == 'O']
print('NO of categorical feature', len(categorical_feature))

discrete_feature = [feature for feature in numerical_features if len(df[feature].unique()) <= 25]
print('No. of discrete Feature', len(discrete_feature))

continuous_feature = [feature for feature in numerical_features if feature not in discrete_feature]
print('No. of Continuous Feature', len(continuous_feature))

No. of Numeric Feature 12
NO of categorical feature 6
No. of discrete Feature 9
No. of Continuous Feature 3


In [48]:
## Train Test Split

X = df.drop(['ProdTaken'], axis=1)
y = df['ProdTaken']

In [49]:
pd.DataFrame(X)

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisiting
0,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [51]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33,random_state=42)
X_train.shape

(3274, 17)

In [52]:
## create column transformer with three type of transformer

numerical_features = X.select_dtypes(exclude='object').columns
categorical_feature = X.select_dtypes(include='object').columns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numerical_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder', oh_transformer, categorical_feature),
        ('StandardScaler', numerical_transformer, numerical_features)
    ]
)

In [53]:
preprocessor

,transformers,"[('OneHotEncoder', ...), ('StandardScaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [54]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

## AdaBoost Technique
# We combine multiple Algorithm

In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, \
                            f1_score, recall_score, roc_curve, roc_auc_score

In [57]:
models = {
    'Logistic Regression' : LogisticRegression(),
    'Random Forest' : RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'Ada Boost' : AdaBoostClassifier(),
    'Decision Tree' : DecisionTreeClassifier()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## Make prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Calculate Train Model Performance
    model_train_accuracy = accuracy_score(y_train, y_train_pred)
    model_f1_train = f1_score(y_train, y_train_pred, average='weighted')
    model_precision_train = precision_score(y_train, y_train_pred)
    model_recall_train = recall_score(y_train,y_train_pred)
    model_roc_auc_train = roc_auc_score(y_train, y_train_pred)

    ## calculate Test Model Performance
    model_test_accuracy = accuracy_score(y_test,y_test_pred)
    model_f1_score = f1_score(y_test,y_test_pred, average='weighted')
    model_precision_score = precision_score(y_test,y_test_pred)
    model_recall_score = recall_score(y_test,y_test_pred)
    model_roc_auc_score = roc_auc_score(y_test,y_test_pred)


    print(list(models.keys())[i])
    print('Model Performance for Training Dataset')
    print('-Accuracy: {:.2f}'.format(model_train_accuracy))
    print('-f1 score: {:.2f}'.format(model_f1_train))
    print('-precision: {:.2f}'.format(model_precision_train))
    print('-recall: {:.2f}'.format(model_recall_train))
    print('-Roc auc: {:.2f}'.format(model_roc_auc_train))

    print('Model Test Performance')
    print('-Accuracy: {:.2f}'.format(model_test_accuracy))
    print('-f1 score: {:.2f}'.format(model_f1_score))
    print('-precision: {:.2f}'.format(model_precision_score))
    print('-recall: {:.2f}'.format(model_recall_score))
    print('-Roc auc: {:.2f}'.format(model_roc_auc_score))

    print('-'*20)
    print('\n')

Logistic Regression
Model Performance for Training Dataset
-Accuracy: 0.84
-f1 score: 0.82
-precision: 0.72
-recall: 0.31
-Roc auc: 0.64
Model Test Performance
-Accuracy: 0.85
-f1 score: 0.82
-precision: 0.65
-recall: 0.31
-Roc auc: 0.64
--------------------


Random Forest
Model Performance for Training Dataset
-Accuracy: 1.00
-f1 score: 1.00
-precision: 1.00
-recall: 1.00
-Roc auc: 1.00
Model Test Performance
-Accuracy: 0.92
-f1 score: 0.91
-precision: 0.94
-recall: 0.61
-Roc auc: 0.80
--------------------


Gradient Boosting
Model Performance for Training Dataset
-Accuracy: 0.90
-f1 score: 0.89
-precision: 0.89
-recall: 0.54
-Roc auc: 0.76
Model Test Performance
-Accuracy: 0.87
-f1 score: 0.85
-precision: 0.77
-recall: 0.41
-Roc auc: 0.69
--------------------


Ada Boost
Model Performance for Training Dataset
-Accuracy: 0.85
-f1 score: 0.82
-precision: 0.82
-recall: 0.29
-Roc auc: 0.64
Model Test Performance
-Accuracy: 0.84
-f1 score: 0.81
-precision: 0.65
-recall: 0.24
-Roc auc: 0.

In [58]:
## HyperParameter Training

rf_params ={
           "max_depth": [5, 8, 15, None, 10],
            "max_features": [5, 7, "auto", 8],
            "min_samples_split": [2, 8, 15, 20],
            "n_estimators": [100, 200, 500, 1000]
}
ada_params = {
    "n_estimators":[50,60,70,80,90],
    "algorithm":['SAMME','SAMME.R']
}

In [59]:
randomcv_models = [
    ('RF', RandomForestClassifier(), rf_params),
    ('AB', AdaBoostClassifier(), ada_params)
]

In [62]:
from sklearn.model_selection import RandomizedSearchCV
model_params = {}

for name,model,params in randomcv_models:
    random = RandomizedSearchCV(estimator = model,
                                param_distributions=params,
                                n_iter=100,
                                cv=3,
                                verbose=3,
                                n_jobs=-1)
    
    random.fit(X_train, y_train)
    model_params[name] = random.best_params_

    for model_name in model_params:
        print(f'---------Best Model {model_name}----------')
        print(model_params[model_name])
        
    

Fitting 3 folds for each of 100 candidates, totalling 300 fits
---------Best Model RF----------
{'n_estimators': 500, 'min_samples_split': 2, 'max_features': 8, 'max_depth': 15}
Fitting 3 folds for each of 10 candidates, totalling 30 fits
---------Best Model RF----------
{'n_estimators': 500, 'min_samples_split': 2, 'max_features': 8, 'max_depth': 15}
---------Best Model AB----------
{'n_estimators': 90, 'algorithm': 'SAMME'}


In [63]:
models = {
    
    'Random Forest' : RandomForestClassifier(n_estimators=500, min_samples_split=2, max_features=8, max_depth=15, n_jobs=-1),
    
    'Ada Boost' : AdaBoostClassifier(n_estimators= 90, algorithm='SAMME')
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## Make prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## Calculate Train Model Performance
    model_train_accuracy = accuracy_score(y_train, y_train_pred)
    model_f1_train = f1_score(y_train, y_train_pred, average='weighted')
    model_precision_train = precision_score(y_train, y_train_pred)
    model_recall_train = recall_score(y_train,y_train_pred)
    model_roc_auc_train = roc_auc_score(y_train, y_train_pred)

    ## calculate Test Model Performance
    model_test_accuracy = accuracy_score(y_test,y_test_pred)
    model_f1_score = f1_score(y_test,y_test_pred, average='weighted')
    model_precision_score = precision_score(y_test,y_test_pred)
    model_recall_score = recall_score(y_test,y_test_pred)
    model_roc_auc_score = roc_auc_score(y_test,y_test_pred)


    print(list(models.keys())[i])
    print('Model Performance for Training Dataset')
    print('-Accuracy: {:.2f}'.format(model_train_accuracy))
    print('-f1 score: {:.2f}'.format(model_f1_train))
    print('-precision: {:.2f}'.format(model_precision_train))
    print('-recall: {:.2f}'.format(model_recall_train))
    print('-Roc auc: {:.2f}'.format(model_roc_auc_train))

    print('Model Test Performance')
    print('-Accuracy: {:.2f}'.format(model_test_accuracy))
    print('-f1 score: {:.2f}'.format(model_f1_score))
    print('-precision: {:.2f}'.format(model_precision_score))
    print('-recall: {:.2f}'.format(model_recall_score))
    print('-Roc auc: {:.2f}'.format(model_roc_auc_score))

    print('-'*20)
    print('\n')

Random Forest
Model Performance for Training Dataset
-Accuracy: 1.00
-f1 score: 1.00
-precision: 1.00
-recall: 1.00
-Roc auc: 1.00
Model Test Performance
-Accuracy: 0.93
-f1 score: 0.92
-precision: 0.92
-recall: 0.65
-Roc auc: 0.82
--------------------


Ada Boost
Model Performance for Training Dataset
-Accuracy: 0.85
-f1 score: 0.83
-precision: 0.77
-recall: 0.33
-Roc auc: 0.65
Model Test Performance
-Accuracy: 0.84
-f1 score: 0.82
-precision: 0.66
-recall: 0.28
-Roc auc: 0.62
--------------------


